# Carte de la dépendance automobile contrainte — prototype (mesure 13)

**Ce que fait ce notebook** : pour chaque commune d'un département, il estime combien d'actifs vont travailler en voiture **sans alternative crédible**, et parmi eux combien sont **employés ou ouvriers**, puis croise avec la pauvreté locale.

| Source | Ce qu'on en tire |
|---|---|
| Insee — MOBPRO 2022 (fichier détail) | qui va travailler où, avec quel mode, quelle catégorie sociale |
| Insee — Filosofi 2021 carroyé 200 m | taux de pauvreté et niveau de vie par carreau, agrégés à la commune |
| transport.data.gouv.fr — GTFS | liaisons directes en car / train entre communes, matin et soir d'un jour ouvré |
| Registre de preuve de covoiturage | trajets covoiturés réellement observés entre communes |
| geo.api.gouv.fr | contours et centres des communes |

**Mode d'emploi**
1. Changez le code du département dans la cellule « Paramètres » ci-dessous.
2. Menu **Exécution → Tout exécuter**. Comptez 5 à 15 minutes selon le département (les gros fichiers régionaux de transport sont les plus longs).
3. À la fin : une carte s'affiche, et un fichier `resultats_DEP.zip` se télécharge. Il contient les fichiers à déposer dans le dossier `data/` de l'app Streamlit.

**Score d'alternative (0 à 3) pour chaque couple commune de résidence → commune de travail** : on garde le meilleur des trois modes.
- *Transport collectif* : 3 = au moins 3 départs le matin, un retour le soir, temps ≤ 1,3 × voiture ; 2 = 2 départs, ≤ 1,8 × ; 1 = au moins un aller-retour possible, ≤ 3 × ; 0 sinon. Seules les liaisons **directes** (sans correspondance) sont comptées.
- *Vélo / VAE / véli* : 3 si ≤ 3 km, 2 si ≤ 8 km, 1 si ≤ 20 km (VAE ou véli), 0 au-delà.
- *Covoiturage* : 2 si ≥ 60 trajets/mois observés sur la paire, 1 si ≥ 10 (plafonné à 2 car la donnée ne couvre que les plateformes).

Un automobiliste est **captif** si le meilleur score de son trajet est 0 ou 1.

## 1. Paramètres

In [ ]:
DEP = "24"                 # code du département étudié (ex. "24" Dordogne, "23" Creuse, "2A" Corse-du-Sud)
MOIS_COVOITURAGE = 2       # nombre de mois récents du registre de covoiturage (0 pour ignorer ; ~300 Mo par mois)
INCLURE_TER = True         # ajouter les horaires TER nationaux en plus des réseaux locaux
GTFS_MAX_MO = 400          # on ignore les fichiers de transport plus lourds que ça
GTFS_URLS_EN_PLUS = []     # liens GTFS à ajouter à la main si un réseau manque (copiés depuis transport.data.gouv.fr)

# Hypothèses du calcul (modifiables)
PARAMS = dict(
    detour_route=1.3,             # distance routière ≈ 1,3 × vol d'oiseau
    vitesse_voiture_kmh=55,       # vitesse moyenne en voiture (rural / périurbain)
    penalite_acces_tc_min=10,     # marche + attente avant le car ou le train
    fenetre_matin=("06:00", "09:30"),   # heure d'arrivée au travail
    fenetre_soir=("16:00", "20:00"),    # heure de départ du retour
    velo_km_3=3, velo_km_2=8, velo_km_1=20,
    covoit_2=60, covoit_1=10,
    jours_travailles=220, conso_l_100=6.5,
    prix_crise=2.37,              # €/L gazole, niveau de septembre 2026
    prix_reference=1.70,          # €/L avant crise (hypothèse)
    score_captif_max=1,
)

## 2. Installation et fonctions de calcul
Rien à modifier ici.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas", "pyarrow", "folium", "mapclassify", "matplotlib", "requests"], check=False)

import os, json, re, time, zipfile, shutil
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import geopandas as gpd

DATA = Path("donnees_brutes"); DATA.mkdir(exist_ok=True)
OUT = Path(f"resultats_{DEP}"); OUT.mkdir(exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (prototype dependance-auto; open data)"}

def get_json(url):
    r = requests.get(url, headers=UA, timeout=120)
    r.raise_for_status()
    return r.json()

def download(url, dest, quiet=False):
    """Télécharge un fichier une seule fois (cache local)."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        if not quiet: print(f"  (déjà téléchargé) {dest.name}")
        return dest
    with requests.get(url, headers=UA, stream=True, timeout=600) as r:
        r.raise_for_status()
        total, done, t0 = int(r.headers.get("content-length", 0)), 0, time.time()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk); done += len(chunk)
        if not quiet:
            print(f"  ✓ {dest.name} ({done/1e6:.0f} Mo en {time.time()-t0:.0f} s)")
    return dest

In [ ]:
# --- Fonctions de calcul (copie de depauto_core.py) ---
# -*- coding: utf-8 -*-
"""
Cœur de calcul — Carte de la dépendance automobile contrainte (mesure 13).

Toutes les fonctions prennent des chemins de fichiers locaux : le téléchargement
est fait à part (voir le notebook), ce qui permet de tester sur des données fictives.
"""
import io
import re
import zipfile
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
from scipy.spatial import cKDTree

# ---------------------------------------------------------------------------
# Paramètres par défaut (modifiables dans le notebook)
# ---------------------------------------------------------------------------
DEFAULTS = dict(
    detour_route=1.3,          # distance routière ≈ 1,3 × vol d'oiseau
    vitesse_voiture_kmh=55,    # vitesse moyenne voiture (rural / périurbain)
    penalite_acces_tc_min=10,  # marche + attente avant le car / train
    fenetre_matin=("06:00", "09:30"),  # arrivée au travail
    fenetre_soir=("16:00", "20:00"),   # départ du retour
    # seuils vélo / VAE / véli (km routiers)
    velo_km_3=3, velo_km_2=8, velo_km_1=20,
    # seuils covoiturage (trajets / mois sur la paire de communes, 2 sens)
    covoit_2=60, covoit_1=10,
    # paramètres d'exposition au prix du carburant
    jours_travailles=220, conso_l_100=6.5,
    prix_crise=2.37, prix_reference=1.70,
    score_captif_max=1,        # un flux est "captif" si score alternative <= 1
)


def hms_to_sec(s):
    """'25:10:00' -> secondes (les GTFS dépassent 24h)."""
    s = s.astype(str).str.strip()
    parts = s.str.split(":", expand=True)
    parts = parts.apply(pd.to_numeric, errors="coerce")
    return parts[0] * 3600 + parts[1] * 60 + parts.get(2, 0).fillna(0)


def _hhmm(t):
    h, m = t.split(":")
    return int(h) * 3600 + int(m) * 60


# ---------------------------------------------------------------------------
# 1. Référentiel communes
# ---------------------------------------------------------------------------
def load_communes(centres_json_path, contours_geojson_path):
    """
    centres_json_path : export geo.api.gouv.fr de toutes les communes (code, nom,
                        centre, surface, population, codeDepartement)
    contours_geojson_path : contours des communes du département étudié
    Retourne (centres : DataFrame toutes communes en Lambert-93, contours : GeoDataFrame)
    """
    raw = pd.read_json(centres_json_path, dtype={"code": str, "codeDepartement": str})
    raw["code"] = raw["code"].astype(str).str.zfill(5)
    raw = raw[raw["centre"].notna()].copy()
    raw["lon"] = raw["centre"].apply(lambda c: c["coordinates"][0])
    raw["lat"] = raw["centre"].apply(lambda c: c["coordinates"][1])
    tr = Transformer.from_crs(4326, 2154, always_xy=True)
    raw["x"], raw["y"] = tr.transform(raw["lon"].values, raw["lat"].values)
    raw["surface_km2"] = raw.get("surface", pd.Series(np.nan, index=raw.index)) / 100.0
    centres = raw[["code", "nom", "lon", "lat", "x", "y", "surface_km2",
                   "population", "codeDepartement"]].reset_index(drop=True)

    contours = gpd.read_file(contours_geojson_path)
    contours["code"] = contours["code"].astype(str).str.zfill(5)
    contours = contours[["code", "nom", "geometry"]].to_crs(4326)
    return centres, contours


# ---------------------------------------------------------------------------
# 2. MOBPRO : flux domicile-travail
# ---------------------------------------------------------------------------
def detect_code_voiture(varmod_path=None):
    """Retrouve le code TRANS 'Voiture' dans le fichier de métadonnées Insee."""
    if varmod_path is None:
        return "5"
    try:
        vm = pd.read_csv(varmod_path, sep=";", dtype=str, encoding="utf-8",
                         on_bad_lines="skip")
        vm.columns = [c.upper() for c in vm.columns]
        t = vm[vm["COD_VAR"] == "TRANS"]
    except Exception as e:
        print(f"Métadonnées illisibles ({e}) : code voiture = 5 par défaut")
        return "5"
    print("Modalités TRANS :")
    for _, r in t.iterrows():
        print(f"   {r['COD_MOD']} = {r['LIB_MOD']}")
    hit = t[t["LIB_MOD"].str.contains("oiture", case=False, na=False)]
    code = hit["COD_MOD"].iloc[0] if len(hit) else "5"
    print(f"→ code voiture retenu : {code}")
    return code


def load_mobpro(parquet_path, dep, code_voiture="5"):
    """
    Lit le fichier détail MOBPRO et agrège les actifs par
    commune de résidence × commune de travail × mode × catégorie modeste.
    Catégorie modeste (proxy individuel) : CS1 = 5 (employés) ou 6 (ouvriers).
    """
    import pyarrow.dataset as ds
    d = ds.dataset(parquet_path, format="parquet")
    cols = [c for c in ["COMMUNE", "DCLT", "TRANS", "IPONDI", "CS1"]
            if c in d.schema.names]
    tbl = d.to_table(columns=cols).to_pandas()
    tbl["COMMUNE"] = tbl["COMMUNE"].astype(str)
    tbl = tbl[tbl["COMMUNE"].str.startswith(dep)].copy()
    tbl["DCLT"] = tbl["DCLT"].astype(str)
    tbl["TRANS"] = tbl["TRANS"].astype(str)
    tbl["IPONDI"] = pd.to_numeric(tbl["IPONDI"], errors="coerce").fillna(0)
    if "CS1" in tbl:
        tbl["modeste"] = tbl["CS1"].astype(str).isin(["5", "6"])
    else:
        tbl["modeste"] = False
    tbl["voiture"] = tbl["TRANS"] == str(code_voiture)

    g = (tbl.groupby(["COMMUNE", "DCLT"])
            .apply(lambda x: pd.Series({
                "actifs": x["IPONDI"].sum(),
                "actifs_voiture": x.loc[x["voiture"], "IPONDI"].sum(),
                "actifs_voiture_modestes": x.loc[x["voiture"] & x["modeste"], "IPONDI"].sum(),
                "actifs_modestes": x.loc[x["modeste"], "IPONDI"].sum(),
            }), include_groups=False)
            .reset_index()
            .rename(columns={"COMMUNE": "orig", "DCLT": "dest"}))
    # part modale existante (pour contrôle)
    modes = tbl.pivot_table(index="COMMUNE", columns="TRANS", values="IPONDI",
                            aggfunc="sum", fill_value=0)
    print(f"MOBPRO : {len(tbl):,} individus-ligne, {g['actifs'].sum():,.0f} actifs, "
          f"{len(g):,} couples de communes")
    return g, modes


# ---------------------------------------------------------------------------
# 3. Filosofi carroyé
# ---------------------------------------------------------------------------
def parse_idcar(ids):
    """'CRS3035RES200mN2029800E4252400' -> (x, y) centre du carreau en EPSG:3035."""
    m = ids.str.extract(r"N(\d+)E(\d+)")
    n = pd.to_numeric(m[0]); e = pd.to_numeric(m[1])
    # taille du carreau lue dans l'identifiant (200m / 1000m)
    res = pd.to_numeric(ids.str.extract(r"RES(\d+)m")[0]).fillna(200)
    return e + res / 2, n + res / 2


def load_filosofi(csv_path_or_zip, dep, contours):
    """
    Carreaux 200 m du département. Rattachement spatial à la commune actuelle
    (évite les écarts de code officiel géographique entre millésimes).
    Retourne (carreaux GeoDataFrame points, agrégats par commune).
    """
    usecols = ["idcar_200m", "lcog_geo", "ind", "men", "men_pauv", "ind_snv", "i_est_200"]
    path = str(csv_path_or_zip)
    if path.endswith(".zip"):
        z = zipfile.ZipFile(path)
        names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        # fichier métropole en priorité
        names = sorted(names, key=lambda n: ("met" not in n.lower(), n))
        print("Filosofi : lecture de", names[0])
        opener = lambda: z.open(names[0])
    else:
        opener = lambda: open(path, "rb")
    with opener() as f:
        sample = f.read(4000).decode("utf-8", "ignore")
    sep = ";" if sample.count(";") > sample.count(",") else ","

    chunks = []
    with opener() as f:
        for ch in pd.read_csv(f, sep=sep, chunksize=500_000, dtype={"lcog_geo": str},
                              low_memory=False):
            ch.columns = [c.lower() for c in ch.columns]
            ch = ch[[c for c in usecols if c in ch.columns]]
            lc = ch["lcog_geo"].astype(str)
            # lcog_geo concatène les codes communes (5 caractères chacun)
            mask = np.zeros(len(ch), dtype=bool)
            for k in range(0, 25, 5):
                mask |= (lc.str[k:k + len(dep)] == dep).values
            chunks.append(ch[mask])
    car = pd.concat(chunks, ignore_index=True)
    x3035, y3035 = parse_idcar(car["idcar_200m"])
    tr = Transformer.from_crs(3035, 4326, always_xy=True)
    car["lon"], car["lat"] = tr.transform(x3035.values, y3035.values)
    car = gpd.GeoDataFrame(car, geometry=gpd.points_from_xy(car["lon"], car["lat"]), crs=4326)
    car = gpd.sjoin(car, contours[["code", "geometry"]], how="inner", predicate="within")
    car = car.drop(columns=["index_right"]).rename(columns={"code": "commune"})
    for c in ["ind", "men", "men_pauv", "ind_snv"]:
        car[c] = pd.to_numeric(car[c], errors="coerce").fillna(0)
    car["taux_pauvrete"] = np.where(car["men"] > 0, car["men_pauv"] / car["men"], np.nan)
    car["niveau_vie"] = np.where(car["ind"] > 0, car["ind_snv"] / car["ind"], np.nan)

    agg = car.groupby("commune").agg(
        pop_filosofi=("ind", "sum"), men=("men", "sum"),
        men_pauv=("men_pauv", "sum"), ind_snv=("ind_snv", "sum"),
        n_carreaux=("idcar_200m", "count")).reset_index()
    agg["taux_pauvrete"] = agg["men_pauv"] / agg["men"].where(agg["men"] > 0)
    agg["niveau_vie_moyen"] = agg["ind_snv"] / agg["pop_filosofi"].where(agg["pop_filosofi"] > 0)
    print(f"Filosofi : {len(car):,} carreaux habités dans {agg.shape[0]} communes")
    return car, agg[["commune", "pop_filosofi", "taux_pauvrete", "niveau_vie_moyen"]]


# ---------------------------------------------------------------------------
# 4. GTFS : liaisons directes en transport collectif entre communes
# ---------------------------------------------------------------------------
def _read_gtfs_table(z, name, usecols=None):
    cands = [n for n in z.namelist() if n.split("/")[-1] == name]
    if not cands:
        return None
    with z.open(cands[0]) as f:
        df = pd.read_csv(f, dtype=str, low_memory=False, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    if usecols:
        df = df[[c for c in usecols if c in df.columns]]
    return df


def _active_services(cal, cald, d):
    """service_id actifs à la date d (objet date)."""
    active = set()
    ds_ = d.strftime("%Y%m%d")
    if cal is not None and len(cal):
        wd = ["monday", "tuesday", "wednesday", "thursday", "friday",
              "saturday", "sunday"][d.weekday()]
        m = (cal["start_date"] <= ds_) & (cal["end_date"] >= ds_) & (cal[wd] == "1")
        active |= set(cal.loc[m, "service_id"])
    if cald is not None and len(cald):
        x = cald[cald["date"] == ds_]
        active |= set(x.loc[x["exception_type"] == "1", "service_id"])
        active -= set(x.loc[x["exception_type"] == "2", "service_id"])
    return active


def _pick_weekday(cal, cald, trips):
    """Choisit le mardi ou jeudi le plus chargé dans la période de validité."""
    dates = []
    if cal is not None and len(cal):
        dates += list(cal["start_date"]) + list(cal["end_date"])
    if cald is not None and len(cald):
        dates += list(cald["date"])
    dates = [datetime.strptime(x, "%Y%m%d").date() for x in dates if re.fullmatch(r"\d{8}", str(x))]
    if not dates:
        return None, set()
    start, end = min(dates), max(dates)
    # on ne regarde pas au-delà de 120 jours après le début (gain de temps)
    end = min(end, start + timedelta(days=180))
    best, best_n, best_set = None, -1, set()
    d = start
    while d <= end:
        if d.weekday() in (1, 3):  # mardi, jeudi
            s = _active_services(cal, cald, d)
            n = trips["service_id"].isin(s).sum()
            if n > best_n:
                best, best_n, best_set = d, n, s
        d += timedelta(days=1)
    return best, best_set


def gtfs_commune_links(gtfs_paths, centres, dep_codes, params=DEFAULTS, max_stop_km=3.0):
    """
    Pour chaque GTFS : rattache les arrêts à la commune la plus proche (≤ max_stop_km
    du centre), choisit un jour ouvré type, puis calcule pour chaque couple de
    communes (o, d) desservi par une même course :
      - nb de départs du matin arrivant à d dans la fenêtre matin
      - meilleur temps à bord le matin (min)
      - nb de retours d→o partant dans la fenêtre du soir
    Seules les courses passant par au moins une commune du département sont gardées.
    """
    kd = cKDTree(centres[["x", "y"]].values)
    tr = Transformer.from_crs(4326, 2154, always_xy=True)
    m0, m1 = (_hhmm(t) for t in params["fenetre_matin"])
    s0, s1 = (_hhmm(t) for t in params["fenetre_soir"])
    dep_codes = set(dep_codes)
    all_pairs = []

    for p in gtfs_paths:
        try:
            z = zipfile.ZipFile(p)
            stops = _read_gtfs_table(z, "stops.txt", ["stop_id", "stop_lat", "stop_lon"])
            trips = _read_gtfs_table(z, "trips.txt", ["trip_id", "service_id"])
            cal = _read_gtfs_table(z, "calendar.txt")
            cald = _read_gtfs_table(z, "calendar_dates.txt")
            if stops is None or trips is None:
                print(f"  ✗ {p} : stops/trips absents"); continue
            stops = stops.dropna(subset=["stop_lat", "stop_lon"])
            sx, sy = tr.transform(stops["stop_lon"].astype(float).values,
                                  stops["stop_lat"].astype(float).values)
            dist, idx = kd.query(np.c_[sx, sy])
            stops["commune"] = centres["code"].values[idx]
            stops.loc[dist > max_stop_km * 1000, "commune"] = None
            stops = stops.dropna(subset=["commune"])
            if not stops["commune"].isin(dep_codes).any():
                print(f"  – {p} : aucun arrêt dans le département, ignoré"); continue

            day, services = _pick_weekday(cal, cald, trips)
            if not services:
                print(f"  ✗ {p} : pas de calendrier exploitable"); continue
            trips = trips[trips["service_id"].isin(services)]

            st = _read_gtfs_table(z, "stop_times.txt",
                                  ["trip_id", "stop_id", "stop_sequence",
                                   "arrival_time", "departure_time"])
            st = st[st["trip_id"].isin(trips["trip_id"])]
            st = st.merge(stops[["stop_id", "commune"]], on="stop_id", how="inner")
            # courses touchant le département
            tdep = st.loc[st["commune"].isin(dep_codes), "trip_id"].unique()
            st = st[st["trip_id"].isin(tdep)].copy()
            if st.empty:
                continue
            st["seq"] = pd.to_numeric(st["stop_sequence"], errors="coerce")
            st["arr"] = hms_to_sec(st["arrival_time"].fillna(st["departure_time"]))
            st["dep"] = hms_to_sec(st["departure_time"].fillna(st["arrival_time"]))
            # par course et commune : premier départ / dernière arrivée
            first = st.sort_values("seq").groupby(["trip_id", "commune"]).agg(
                seq_min=("seq", "min"), seq_max=("seq", "max"),
                dep=("dep", "first"), arr=("arr", "last")).reset_index()
            o = first.rename(columns={"commune": "orig", "seq_min": "so", "dep": "t_dep"})[
                ["trip_id", "orig", "so", "t_dep"]]
            d = first.rename(columns={"commune": "dest", "seq_max": "sd", "arr": "t_arr"})[
                ["trip_id", "dest", "sd", "t_arr"]]
            pr = o.merge(d, on="trip_id")
            pr = pr[(pr["orig"] != pr["dest"]) & (pr["so"] < pr["sd"])]
            pr["duree_min"] = (pr["t_arr"] - pr["t_dep"]) / 60
            pr = pr[pr["duree_min"] > 0]
            pr["matin"] = pr["t_arr"].between(m0, m1)
            pr["soir"] = pr["t_dep"].between(s0, s1)
            pr["src"] = str(p).split("/")[-1]
            all_pairs.append(pr[["orig", "dest", "trip_id", "duree_min", "matin", "soir", "src"]])
            print(f"  ✓ {str(p).split('/')[-1]} : jour type {day}, "
                  f"{len(tdep):,} courses, {len(pr):,} liaisons course×couple")
        except Exception as e:  # un GTFS cassé ne doit pas bloquer le reste
            print(f"  ✗ {p} : {e}")

    if not all_pairs:
        return pd.DataFrame(columns=["orig", "dest", "tc_departs_matin", "tc_duree_min",
                                     "tc_retours_soir"])
    pr = pd.concat(all_pairs, ignore_index=True)
    matin = (pr[pr["matin"]].groupby(["orig", "dest"])
             .agg(tc_departs_matin=("trip_id", "nunique"), tc_duree_min=("duree_min", "min")))
    soir = (pr[pr["soir"]].groupby(["orig", "dest"]).agg(n_soir=("trip_id", "nunique"))
            .reset_index().rename(columns={"orig": "dest", "dest": "orig"})
            .set_index(["orig", "dest"]))
    out = matin.join(soir, how="left").fillna({"n_soir": 0}).reset_index()
    out = out.rename(columns={"n_soir": "tc_retours_soir"})
    return out


# ---------------------------------------------------------------------------
# 5. Registre de preuve de covoiturage
# ---------------------------------------------------------------------------
def load_covoiturage(csv_paths, dep):
    """Nombre de trajets covoiturés par couple de communes (les 2 sens cumulés), par mois."""
    frames, n_mois = [], 0
    for p in csv_paths:
        with open(p, encoding="utf-8", errors="ignore") as f:
            head = f.readline()
        sep = ";" if head.count(";") > head.count(",") else ","
        cols = [c.strip().strip('"') for c in head.split(sep)]
        cs = next((c for c in cols if "start" in c and "insee" in c), None)
        ce = next((c for c in cols if "end" in c and "insee" in c), None)
        if not cs or not ce:
            print(f"  ✗ {p} : colonnes INSEE introuvables"); continue
        n_mois += 1
        for ch in pd.read_csv(p, sep=sep, usecols=[cs, ce], dtype=str, chunksize=1_000_000):
            ch = ch.dropna()
            ch = ch[ch[cs].str.startswith(dep) | ch[ce].str.startswith(dep)]
            frames.append(ch.rename(columns={cs: "a", ce: "b"}))
    if not frames:
        return pd.DataFrame(columns=["orig", "dest", "covoit_mois"])
    c = pd.concat(frames)
    # paire non orientée
    c["k1"] = c[["a", "b"]].min(axis=1); c["k2"] = c[["a", "b"]].max(axis=1)
    g = c.groupby(["k1", "k2"]).size().reset_index(name="n")
    g["covoit_mois"] = g["n"] / max(n_mois, 1)
    g2 = pd.concat([g.rename(columns={"k1": "orig", "k2": "dest"}),
                    g.rename(columns={"k2": "orig", "k1": "dest"})])
    print(f"Covoiturage : {len(c):,} trajets touchant le département sur {n_mois} mois")
    return g2[["orig", "dest", "covoit_mois"]].drop_duplicates(["orig", "dest"])


# ---------------------------------------------------------------------------
# 6. Score d'alternative par flux (mesure 9) et indicateur communal (mesure 13)
# ---------------------------------------------------------------------------
def score_flows(flows, centres, tc, covoit, params=DEFAULTS):
    p = params
    c = centres.set_index("code")
    f = flows.copy()
    known = f["orig"].isin(c.index) & f["dest"].isin(c.index)
    n_unk = f.loc[~known, "actifs"].sum()
    if n_unk:
        print(f"  ⚠ {n_unk:,.0f} actifs vers des communes hors référentiel "
              f"(étranger, code fusionné…) exclus du score")
    f = f[known].copy()
    xo, yo = c.loc[f["orig"], "x"].values, c.loc[f["orig"], "y"].values
    xd, yd = c.loc[f["dest"], "x"].values, c.loc[f["dest"], "y"].values
    crow = np.hypot(xo - xd, yo - yd) / 1000
    intra = f["orig"].values == f["dest"].values
    surf = c.loc[f["orig"], "surface_km2"].fillna(10).values
    crow = np.where(intra, 0.4 * np.sqrt(surf), crow)
    f["km_route"] = np.maximum(crow * p["detour_route"], 0.5)
    f["voiture_min"] = f["km_route"] / p["vitesse_voiture_kmh"] * 60 + 3

    f = f.merge(tc, on=["orig", "dest"], how="left").merge(covoit, on=["orig", "dest"], how="left")
    f[["tc_departs_matin", "tc_retours_soir", "covoit_mois"]] = \
        f[["tc_departs_matin", "tc_retours_soir", "covoit_mois"]].fillna(0)

    # --- transport collectif
    ratio = (f["tc_duree_min"] + p["penalite_acces_tc_min"]) / f["voiture_min"]
    f["tc_ratio"] = ratio.round(2)
    ok = (f["tc_departs_matin"] >= 1) & (f["tc_retours_soir"] >= 1)
    f["score_tc"] = 0
    f.loc[ok & (ratio <= 3.0), "score_tc"] = 1
    f.loc[ok & (ratio <= 1.8) & (f["tc_departs_matin"] >= 2), "score_tc"] = 2
    f.loc[ok & (ratio <= 1.3) & (f["tc_departs_matin"] >= 3), "score_tc"] = 3
    # --- vélo / VAE / véli
    km = f["km_route"]
    f["score_velo"] = np.select([km <= p["velo_km_3"], km <= p["velo_km_2"], km <= p["velo_km_1"]],
                                [3, 2, 1], 0)
    # --- covoiturage (plafonné à 2 : donnée partielle)
    f["score_covoit"] = np.select([f["covoit_mois"] >= p["covoit_2"], f["covoit_mois"] >= p["covoit_1"]],
                                  [2, 1], 0)
    f["score_alternative"] = f[["score_tc", "score_velo", "score_covoit"]].max(axis=1)
    f["alternative_principale"] = f[["score_tc", "score_velo", "score_covoit"]].idxmax(axis=1) \
        .str.replace("score_", "").where(f["score_alternative"] > 0, "aucune")

    # --- exposition carburant (par actif voiture)
    litres = f["km_route"] * 2 * p["jours_travailles"] * p["conso_l_100"] / 100
    f["litres_an_par_actif"] = litres.round(0)
    f["surcout_crise_an_par_actif"] = (litres * (p["prix_crise"] - p["prix_reference"])).round(0)
    f["captif"] = f["score_alternative"] <= p["score_captif_max"]
    return f


def commune_indicators(f, filo, centres, contours, dep):
    cap = f[f["captif"]]
    g = f.groupby("orig").agg(actifs=("actifs", "sum"),
                              actifs_voiture=("actifs_voiture", "sum"),
                              actifs_modestes=("actifs_modestes", "sum"))
    cols_c = ["captifs", "captifs_modestes", "km_route_moyen_captifs", "surcout_crise_total"]
    gc = pd.DataFrame(columns=cols_c) if cap.empty else cap.groupby("orig").apply(lambda x: pd.Series({
        "captifs": x["actifs_voiture"].sum(),
        "captifs_modestes": x["actifs_voiture_modestes"].sum(),
        "km_route_moyen_captifs": np.average(x["km_route"], weights=x["actifs_voiture"])
            if x["actifs_voiture"].sum() > 0 else np.nan,
        "surcout_crise_total": (x["surcout_crise_an_par_actif"] * x["actifs_voiture"]).sum(),
    }), include_groups=False)
    # score moyen d'alternative pondéré par les automobilistes
    sm = f.groupby("orig").apply(lambda x: np.average(x["score_alternative"], weights=x["actifs_voiture"])
                                 if x["actifs_voiture"].sum() > 0 else np.nan, include_groups=False)
    ind = g.join(gc, how="left").fillna({"captifs": 0, "captifs_modestes": 0,
                                         "surcout_crise_total": 0})
    ind["score_alt_moyen"] = sm
    ind["part_voiture"] = ind["actifs_voiture"] / ind["actifs"]
    ind["part_captifs"] = ind["captifs"] / ind["actifs"]
    ind["part_captifs_modestes"] = ind["captifs_modestes"] / ind["actifs"]
    ind["surcout_moyen_par_captif"] = ind["surcout_crise_total"] / ind["captifs"].where(ind["captifs"] > 0)
    ind = ind.reset_index().rename(columns={"orig": "code"})
    ind = ind.merge(filo.rename(columns={"commune": "code"}), on="code", how="left")

    # indice de priorité 0–100 : rangs en centiles, transparent et réglable
    r = lambda s: s.rank(pct=True)
    ind["indice_priorite"] = (100 * (0.5 * r(ind["part_captifs_modestes"].fillna(0))
                                     + 0.3 * r(ind["taux_pauvrete"].fillna(ind["taux_pauvrete"].median()))
                                     + 0.2 * r(ind["km_route_moyen_captifs"].fillna(0)))).round(1)
    gdf = contours.merge(ind, on="code", how="left")
    gdf = gdf[gdf["code"].str.startswith(dep)]
    return gdf

## 3. Référentiel des communes (geo.api.gouv.fr)

In [ ]:
geo = "https://geo.api.gouv.fr"
info_dep = get_json(f"{geo}/departements/{DEP}?fields=nom,codeRegion")
print("Département :", info_dep["nom"], "— région", info_dep["codeRegion"])

centres_path = DATA / "communes_centres.json"
if not centres_path.exists():
    json.dump(get_json(f"{geo}/communes?fields=code,nom,centre,surface,population,codeDepartement&format=json"),
              open(centres_path, "w"))
contours_path = DATA / f"communes_contours_{DEP}.geojson"
if not contours_path.exists():
    json.dump(get_json(f"{geo}/departements/{DEP}/communes?format=geojson&geometry=contour&fields=code,nom"),
              open(contours_path, "w"))
centres, contours = load_communes(centres_path, contours_path)
print(f"{len(centres):,} communes en France, {len(contours)} dans le département")

## 4. Flux domicile-travail (Insee MOBPRO 2022)
Fichier national de 66 Mo, filtré sur le département.

In [ ]:
mobpro_path = download("https://www.insee.fr/fr/statistiques/fichier/8589904/RP2022_mobpro.parquet",
                       DATA / "RP2022_mobpro.parquet")
varmod_path = download("https://www.insee.fr/fr/statistiques/fichier/8589904/varmod_mobpro_2022.csv",
                       DATA / "varmod_mobpro_2022.csv")
code_voiture = detect_code_voiture(varmod_path)
flows, modes = load_mobpro(mobpro_path, DEP, code_voiture)

## 5. Revenus carroyés (Insee Filosofi 2021, carreaux de 200 m)
Fichier national de 87 Mo. Les carreaux sont rattachés à la commune par leur position.

In [ ]:
filo_zip = download("https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_csv.zip",
                    DATA / "Filosofi2021_carreaux_200m_csv.zip")
carreaux, filo = load_filosofi(filo_zip, DEP, contours)

## 6. Offre de transport collectif (GTFS de transport.data.gouv.fr)
On cherche automatiquement les réseaux qui couvrent le département, sa région, ses intercommunalités (et les TER si demandé).

In [ ]:
def find_gtfs(dep, code_region, inclure_ter=True, max_mo=400):
    epcis = {c.get("codeEpci") for c in get_json(f"{geo}/departements/{dep}/communes?fields=codeEpci")}
    epcis.discard(None)
    datasets = get_json("https://transport.data.gouv.fr/api/datasets")
    found = []
    for ds in datasets:
        if ds.get("type") not in (None, "public-transit"):
            continue
        areas = ds.get("covered_area") or []
        if isinstance(areas, dict):
            areas = [areas]
        def match(a):
            t, code = a.get("type"), str(a.get("insee", ""))
            return ((t == "region" and code == code_region) or
                    (t in ("departement", "department") and code == dep) or
                    (t == "epci" and code in epcis) or
                    (t == "commune" and code.startswith(dep)))
        ok = any(match(a) for a in areas if isinstance(a, dict))
        title = ds.get("title", "")
        if not ok and inclure_ter and re.search(r"\bTER\b", title):
            ok = True
        if not ok:
            continue
        for r in ds.get("resources", []):
            if r.get("format") != "GTFS" or r.get("is_available") is False:
                continue
            size = (r.get("filesize") or 0) / 1e6
            if size and size > max_mo:
                print(f"  – ignoré (trop lourd, {size:.0f} Mo) : {title}")
                continue
            found.append(dict(titre=title, ressource=r.get("title"), url=r.get("url") or r.get("original_url"),
                              mo=round(size, 1)))
    return pd.DataFrame(found).drop_duplicates("url")

gtfs_list = find_gtfs(DEP, info_dep["codeRegion"], INCLURE_TER, GTFS_MAX_MO)
for u in GTFS_URLS_EN_PLUS:
    gtfs_list.loc[len(gtfs_list)] = dict(titre="ajout manuel", ressource=u, url=u, mo=None)
print(f"{len(gtfs_list)} fichiers GTFS candidats :")
display(gtfs_list)

In [ ]:
gtfs_paths = []
for i, row in gtfs_list.reset_index(drop=True).iterrows():
    try:
        gtfs_paths.append(download(row["url"], DATA / f"gtfs_{DEP}_{i:02d}.zip"))
    except Exception as e:
        print(f"  ✗ {row['titre']} : {e}")
print("\nAnalyse des horaires (jour ouvré type) :")
tc = gtfs_commune_links(gtfs_paths, centres, contours["code"], PARAMS)
print(f"→ {len(tc):,} couples de communes reliés en direct le matin")

## 7. Covoiturage observé (registre de preuve de covoiturage)

In [ ]:
def find_rpc(n):
    ds = get_json("https://www.data.gouv.fr/api/1/datasets/5e8ee97c16601da4ee24ffb7/")
    res = []
    for r in ds.get("resources", []):
        if (r.get("format") or "").lower() != "csv":
            continue
        m = re.search(r"(20\d\d)[-_]?(\d\d)", (r.get("title") or "") + " " + (r.get("url") or ""))
        if m:
            res.append((m.group(1) + "-" + m.group(2), r["url"], r.get("title")))
    res.sort(reverse=True)
    return res[:n]

rpc_paths = []
if MOIS_COVOITURAGE > 0:
    for mois, url, titre in find_rpc(MOIS_COVOITURAGE):
        print("Mois", mois)
        try:
            rpc_paths.append(download(url, DATA / f"rpc_{mois}.csv"))
        except Exception as e:
            print(f"  ✗ {titre} : {e}")
covoit = load_covoiturage(rpc_paths, DEP) if rpc_paths else pd.DataFrame(columns=["orig", "dest", "covoit_mois"])

## 8. Score d'alternative par trajet et indicateurs par commune

In [ ]:
flux = score_flows(flows, centres, tc, covoit, PARAMS)
communes = commune_indicators(flux, filo, centres, contours, DEP)

tot_v = flux["actifs_voiture"].sum()
cap = flux.loc[flux["captif"], "actifs_voiture"].sum()
cap_m = flux.loc[flux["captif"], "actifs_voiture_modestes"].sum()
print(f"Actifs résidents : {flux['actifs'].sum():,.0f}")
print(f"  dont en voiture : {tot_v:,.0f} ({tot_v/flux['actifs'].sum():.0%})")
print(f"  dont captifs (aucune alternative crédible) : {cap:,.0f} ({cap/tot_v:.0%} des automobilistes)")
print(f"  dont captifs employés ou ouvriers : {cap_m:,.0f}")
print(f"Surcoût carburant annuel des captifs au prix de crise : "
      f"{communes['surcout_crise_total'].sum()/1e6:,.1f} M€")
print("\nRépartition des automobilistes par score d'alternative :")
print((flux.groupby("score_alternative")["actifs_voiture"].sum() / tot_v).map("{:.0%}".format).to_string())

## 9. Carte

In [ ]:
import folium
tip = ["nom", "actifs", "part_voiture", "part_captifs", "part_captifs_modestes",
       "taux_pauvrete", "km_route_moyen_captifs", "surcout_moyen_par_captif", "indice_priorite"]
show = communes.copy()
for c in ["part_voiture", "part_captifs", "part_captifs_modestes", "taux_pauvrete"]:
    show[c] = (show[c] * 100).round(1)
for c in ["actifs", "km_route_moyen_captifs", "surcout_moyen_par_captif"]:
    show[c] = show[c].round(0)
m = show.explore(column="indice_priorite", cmap="YlOrRd", tooltip=tip, name="Indice de priorité (0–100)",
                 style_kwds=dict(weight=0.4, color="#555"), legend=True)
show.explore(m=m, column="part_captifs_modestes", cmap="PuRd", tooltip=tip, show=False,
             name="% actifs captifs employés/ouvriers", style_kwds=dict(weight=0.4, color="#555"), legend=False)
show.explore(m=m, column="taux_pauvrete", cmap="Blues", tooltip=tip, show=False,
             name="% ménages pauvres (Filosofi)", style_kwds=dict(weight=0.4, color="#555"), legend=False)
folium.LayerControl(collapsed=False).add_to(m)
m.save(OUT / f"carte_{DEP}.html")
m

In [ ]:
print("Les 15 communes prioritaires :")
cols = ["code", "nom", "actifs", "captifs", "captifs_modestes", "part_captifs_modestes",
        "taux_pauvrete", "km_route_moyen_captifs", "surcout_moyen_par_captif", "indice_priorite"]
display(communes.sort_values("indice_priorite", ascending=False)[cols].head(15).round(2))

## 10. Export pour l'app Streamlit
Le zip contient `communes.geojson`, `flux.csv`, `carreaux.csv`, `meta.json` et la carte HTML.
Déposez les 4 premiers fichiers dans le dossier `data/` du dépôt GitHub de l'app.

In [ ]:
communes.to_file(OUT / "communes.geojson", driver="GeoJSON")
cols_f = ["orig", "dest", "actifs", "actifs_voiture", "actifs_voiture_modestes", "km_route", "voiture_min",
          "tc_departs_matin", "tc_retours_soir", "tc_duree_min", "tc_ratio", "covoit_mois",
          "score_tc", "score_velo", "score_covoit", "score_alternative", "alternative_principale",
          "litres_an_par_actif", "surcout_crise_an_par_actif", "captif"]
noms = centres.set_index("code")["nom"]
fx = flux[cols_f].copy()
fx.insert(1, "orig_nom", fx["orig"].map(noms)); fx.insert(3, "dest_nom", fx["dest"].map(noms))
fx.round(2).to_csv(OUT / "flux.csv", index=False)
carreaux[["idcar_200m", "commune", "lon", "lat", "men", "ind", "taux_pauvrete", "niveau_vie"]] \
    .round(5).to_csv(OUT / "carreaux.csv", index=False)
json.dump(dict(departement=DEP, nom=info_dep["nom"], genere_le=pd.Timestamp.now().isoformat(timespec="minutes"),
               params=PARAMS, gtfs=gtfs_list["titre"].tolist(), mois_covoiturage=len(rpc_paths),
               sources=["Insee MOBPRO 2022 (fichier détail)", "Insee Filosofi 2021 carroyé 200 m",
                        "transport.data.gouv.fr (GTFS)", "Registre de preuve de covoiturage",
                        "geo.api.gouv.fr"]),
          open(OUT / "meta.json", "w"), ensure_ascii=False, indent=1)

zip_name = shutil.make_archive(f"resultats_{DEP}", "zip", OUT)
print("Fichiers prêts :", sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Hors Colab : le zip est ici →", zip_name)